# Зачёт. Заказы службы доставки

Сегодня вы работаете аналитиком службы доставки. Нужно проверить журнал заказов, выполнить расчёты и оценить время доставки нового заказа.

**Один общий ноутбук. Уровень назначает преподаватель.** Средний уровень выполняет все пункты со знаком 🔹. Высокий уровень выполняет их и дополнения «Высокий уровень» в каждом блоке.

Работайте в существующих ячейках. В трёх Markdown-ячейках **ВЫВОДЫ** напишите собственные короткие объяснения по полученным результатам. Датасет **deliveries_exam.csv** должен лежать рядом с этим ноутбуком. Это синтетические учебные данные, одна строка описывает запись о заказе. В таблице есть ошибки и повторения.

На выполнение предусмотрен час. Порядок промежуточного показа и сдачи сообщает преподаватель. На рабочем компьютере открыт JupyterLab. Памятка доступна ниже в этом же ноутбуке.

Перед сдачей выполните все свои ячейки сверху вниз, сохраните таблицы и график внутри **solution.ipynb**. Путь сдачи в репозитории — **exam/solution.ipynb**. После окончания работы доступ для загрузки в GitHub открывается по указанию преподавателя.

## Данные студента

Запишите ФИО и назначенный преподавателем уровень в следующей ячейке.

In [35]:
student_name = "Плюснин Илья Александрович"
level = "средний"  # "средний" или "высокий" — по назначению преподавателя


## Поля и единицы

| Столбец | Значение |
|---|---|
| номер_заказа | Идентификатор заказа. Номер требуется для проверки повторов. |
| вид_доставки | обычная или экспресс |
| район | Район доставки. Текст может содержать разные регистры и пробелы. |
| оформлен | Дата и время оформления, формат ДД.ММ.ГГГГ ЧЧ:ММ |
| доставлен | Дата и время завершения. У отменённого заказа может отсутствовать. |
| расстояние_км | Расстояние в километрах |
| количество | Положительное целое число товаров |
| цена_товара | Цена одной единицы товара в рублях, иногда записана текстом |
| скидка_проц | Скидка на товары в процентах от 0 до 100 |
| тариф_км | Стоимость одного километра доставки в рублях |
| базовая_плата | Фиксированная часть стоимости доставки в рублях |
| статус | доставлен или отменён; написание может содержать пробелы и разный регистр |

Стоимость товаров после скидки = количество × цена товара × (1 − скидка_проц / 100).
Стоимость доставки = базовая_плата + расстояние_км × тариф_км.
Итог заказа = стоимость товаров после скидки + стоимость доставки.
Скидка применяется к товарам. Время доставки считаем в минутах.

## 🟨❗ Подключение инструментов

Следующая ячейка подключает Pandas для таблиц, NumPy для числовых операций, Matplotlib для рисунка и scikit-learn для подготовки, обучения и оценки моделей. Здесь также включается удобная точность показа чисел. Библиотеки должны быть установлены на компьютере до начала зачёта.

In [36]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 12})
print("Библиотеки подключены. Рабочая папка:", Path.cwd())


Библиотеки подключены. Рабочая папка: /content


## 🟨❗ Памятка по синтаксису

Примеры ниже используют условные названия столбцов. Их нужно адаптировать к своей задаче.

<details><summary><b>Таблицы, строки, пропуски и условия</b></summary>

~~~python
table = pd.read_csv("file.csv")
table.shape
table.dtypes
table.head()
table.isna().sum()
table["текст"] = table["текст"].astype("string").str.strip().str.lower()

text = table["значение"].astype("string")
text = text.str.replace(" ", "", regex=False)
text = text.str.replace("₽", "", regex=False).str.replace(",", ".", regex=False)
table["значение"] = pd.to_numeric(text, errors="coerce")

table.loc[table.duplicated(keep=False)]
table.duplicated().sum()
table = table.drop_duplicates().copy()
table.duplicated(subset=["код"], keep=False)

mask = (table["число"] < 0) | table["число"].isna()
table.loc[mask]
table = table.loc[~mask].copy()
table["число"].mod(1).ne(0)
table["число"].between(0, 100)
table["текст"] = table["текст"].fillna("неизвестно")
table["новое"] = table["a"] * table["b"]
table["категория"] = np.where((table["a"] > 5) & (table["b"] > 2), "да", "нет")
~~~

**isna** находит пропуски; **coerce** превращает нераспознанный текст в пропуск.
**loc** выбирает строки по условию; **~** меняет выбор на противоположный.
**copy** создаёт отдельную рабочую таблицу. **mod(1)** даёт дробную часть.

</details>

<details><summary><b>Даты, сводки и графики</b></summary>

~~~python
table["начало"] = pd.to_datetime(
    table["начало"], format="%d.%m.%Y %H:%M", errors="coerce"
)
table["минуты"] = (table["конец"] - table["начало"]).dt.total_seconds() / 60
table["час"] = table["начало"].dt.hour

summary = table.groupby("группа").agg(
    записей=("код", "size"),
    среднее=("значение", "mean"),
    медиана=("значение", "median"),
)
summary[["среднее", "медиана"]].plot.bar()
plt.ylabel("Единицы показателя")
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()

table.sort_values("ошибка", ascending=False).head(3)
~~~

**groupby** собирает строки в группы; **agg** задаёт расчёты по группам.
**mean** — среднее, **median** — медиана, **size** — число строк.
**dt** даёт доступ к компонентам даты и разницы дат.

</details>

<details><summary><b>Разделение, подготовка, модели и ошибки</b></summary>

~~~python
train, valid = train_test_split(table, test_size=0.25, random_state=42)

imputer = SimpleImputer(strategy="mean")
imputer.fit(train[feature_names])
x_train = imputer.transform(train[feature_names])
x_valid = imputer.transform(valid[feature_names])

model = LinearRegression()
model.fit(x_train, train[target_name])
prediction = model.predict(x_valid)

other_model = DecisionTreeRegressor(max_depth=4, random_state=42)
other_model.fit(x_train, train[target_name])

mae = mean_absolute_error(valid[target_name], prediction)
rmse = np.sqrt(mean_squared_error(valid[target_name], prediction))
~~~

**train_test_split** один раз разделяет строки на обучение и проверку.
**SimpleImputer.fit** запоминает средние обучающих признаков, **transform** применяет эти средние.
**LinearRegression.fit** находит коэффициенты линейного прогноза.
**DecisionTreeRegressor.fit** подбирает условия для прогноза числа. **max_depth=4** ограничивает путь четырьмя условиями.
**predict** выдаёт прогнозы по готовой модели.
**MAE** — средний абсолютный промах; **RMSE** — корень из средней квадратичной ошибки. Обе метрики имеют единицы целевого показателя.

</details>

## 🔹 1. Загрузка и осмотр

Прочитайте весь CSV в **raw_data**. Покажите размеры, первые строки и типы столбцов.

Выберите рабочую таблицу **data** через отдельную копию:
- средний уровень — только **вид_доставки = обычная**;
- высокий уровень — все строки.

Покажите размер своей рабочей таблицы. Все дальнейшие задания выполняйте на ней.

In [45]:
import pandas as pd


raw_data = pd.read_csv("deliveries_exam.csv")

print(raw_data.shape)
print(raw_data.head())
print(raw_data.dtypes)


data = raw_data[raw_data["вид_доставки"] == "обычная"].copy()




(6040, 12)
  номер_заказа вид_доставки    район          оформлен         доставлен  \
0      D005530     экспресс      ЮГ   09.10.2026 08:50  09.10.2026 09:14   
1      D000711      обычная    запад  01.10.2026 15:17               NaN   
2      D004924     экспресс    центр  13.10.2026 13:53  13.10.2026 14:33   
3      D002154      обычная   восток  18.10.2026 13:05               NaN   
4      D001273      обычная   Запад   04.10.2026 22:32  04.10.2026 23:21   

   расстояние_км  количество цена_товара  скидка_проц  тариф_км  \
0            5.2         7.0    122,35 ₽         20.0        26   
1            8.3         6.0     221,89           5.0        18   
2           13.4         6.0    564,57 ₽          0.0        26   
3            8.0         2.0      77,38          15.0        18   
4           13.6         5.0    407,31 ₽          5.0        18   

   базовая_плата     статус  
0            100  доставлен  
1             70    отменён  
2            100  доставлен  
3        

## 🔹 2. Очистка

1. Приведите район и статус к нижнему регистру, уберите крайние пробелы. Неизвестный район обозначьте «неизвестно».
2. Переведите цену товара из текста в число. Пропуски цены сохраните неизвестными. Покажите пропуски в таблице.
3. Покажите полные повторы перед удалением, посчитайте лишние копии, удалите их.
4. Покажите и исключите строки с известным неположительным расстоянием и с неизвестным, неположительным или дробным количеством товаров. Неизвестное расстояние оставьте для подготовки модели. Покажите размеры после обработки.

**Высокий уровень — дополнительно**
- Покажите повторяющиеся номера с разными записями, отложите все версии таких номеров.
- Исключите известные неположительные цены и скидки вне 0–100.
- Преобразуйте обе даты с обработкой ошибок. Покажите и исключите строки, где известное время завершения раньше известного времени оформления.

Выполняйте проверки по условиям. Большая положительная цена или долгая доставка сами по себе не являются основанием удалить строку.

In [44]:
raw_data["statys"] = raw_data['район'].str.strip().str.lower()
raw_data['район'] = raw_data["statys"].str.strip().str.lower()





(6040, 12)
  номер_заказа вид_доставки    район          оформлен         доставлен  \
0      D005530     экспресс      ЮГ   09.10.2026 08:50  09.10.2026 09:14   
1      D000711      обычная    запад  01.10.2026 15:17               NaN   
2      D004924     экспресс    центр  13.10.2026 13:53  13.10.2026 14:33   
3      D002154      обычная   восток  18.10.2026 13:05               NaN   
4      D001273      обычная   Запад   04.10.2026 22:32  04.10.2026 23:21   

   расстояние_км  количество цена_товара  скидка_проц  тариф_км  \
0            5.2         7.0    122,35 ₽         20.0        26   
1            8.3         6.0     221,89           5.0        18   
2           13.4         6.0    564,57 ₽          0.0        26   
3            8.0         2.0      77,38          15.0        18   
4           13.6         5.0    407,31 ₽          5.0        18   

   базовая_плата     статус  
0            100  доставлен  
1             70    отменён  
2            100  доставлен  
3        

### ❓ ВЫВОДЫ — данные

Коротко назовите, что вы удалили, чем заполнили пропуски и что оставили неизвестным. Объясните одно решение по обработке данных в 2–3 предложениях.

**Мой ответ:** …

## 🔹 3. Расчётные столбцы

Преобразуйте обе даты в даты и время с обработкой нераспознанных значений. Если они уже преобразованы, используйте готовые столбцы.

Создайте:
- **сумма_после_скидки** по формуле из описания;
- **длительность_мин** как разность времени завершения и оформления в минутах;
- **час_заказа** как час оформления.

Покажите несколько строк с исходными и рассчитанными значениями. Неизвестная цена даёт неизвестную сумму.

**Высокий уровень — дополнительно**
- Рассчитайте **стоимость_доставки** и **итого**.
- Создайте **крупный_дальний** со значениями «да»/«нет». «Да» означает одновременно **итого ≥ 2500 ₽** и **расстояние_км ≥ 8**. Для неизвестного итога или расстояния оставьте категорию «неизвестно».

## 🔹 4. Аналитика и один график

Создайте **analysis_data** — копию строк, где нормализованный статус «доставлен» и известна положительная длительность.

По районам рассчитайте:
- число заказов;
- среднюю и медианную длительность;
- среднюю сумму после скидки.

Постройте один график, по которому можно сравнить среднюю и медианную длительность районов. Подпишите оси, единицы и легенду.

**Высокий уровень — дополнительно**
Сравните среднюю длительность районов отдельно для расстояний **меньше 8 км** и **от 8 км**. Для этого сравнения используйте известные расстояния. Проверьте, одинаков ли вывод о районах в двух группах.

In [41]:
# Ваш код для задания 4
pass


### ❓ ВЫВОДЫ — аналитика

Сравните два района с конкретными числами из своей таблицы. Почему среднее и медиана могут отличаться? Высокий уровень также комментирует сравнение внутри двух групп расстояний. Достаточно 2–4 предложений.

**Мой ответ:** …

## 🔹 5. Прогноз длительности

Работайте с **analysis_data**. Целевой столбец — **длительность_мин**. Используйте три входа в таком порядке:
**расстояние_км, количество, час_заказа**.

1. Один раз разделите строки на обучение и проверку — **test_size=0.25, random_state=42**.
2. Обучите SimpleImputer со стратегией mean только на обучающих входах и преобразуйте обе части.
3. Для простого ориентира всем проверочным строкам предскажите среднюю длительность обучения.
4. Обучите LinearRegression и получите прогнозы для проверки.
5. Выведите таблицу MAE и RMSE ориентира и регрессии в минутах.

**Высокий уровень — дополнительно**
Обучите **DecisionTreeRegressor(max_depth=4, random_state=42)** на тех же обучающих данных. Добавьте его MAE и RMSE в сравнение. Проверочные заказы должны быть одинаковыми для всех подходов.

In [43]:
# Ваш код для задания 5
pass


## 🔹 6. Новый заказ и итог

Создайте новую строку с тремя входами в том же порядке — **7.5 км, 3 товара, час оформления 18**. Примените готовый imputer и обученную регрессию. Покажите прогноз в минутах.

**Высокий уровень — дополнительно**
- Получите прогноз дерева для той же строки.
- Из двух моделей выберите лучшую по MAE. Для неё покажите три проверочных заказа с наибольшей абсолютной ошибкой — номер, факт, прогноз, абсолютный промах.

Заполните итоговый вывод ниже.

In [42]:
# Ваш код для задания 6
pass


### ❓ ВЫВОДЫ — модель

Что означает MAE вашей модели в минутах? Где качество проверено и почему для новой заявки ошибка пока неизвестна? Назовите один столбец исходной таблицы, который опасно включать во входы для прогноза в момент оформления, и объясните причину. Высокий уровень также кратко сравнивает две модели и крупнейшие промахи. Достаточно 3–5 предложений.

**Мой ответ:** …